# 🎬 The Real Houseboys of San Francisco - Automated Confessionals Pipeline

This notebook automatically animates all 18 confessional audio clips with their character portraits to create talking-head reality TV confessionals.

### ⚠️ Before running:
- **On Kaggle**: In the right sidebar: **Settings** -> **Accelerator** -> select **GPU T4 x2** (or GPU T4). Set **Internet** -> **Internet On**.
- **On Google Colab**: **Runtime** -> **Change runtime type** -> select **T4 GPU**.
- Click **Runtime -> Run all** (or press Shift+Enter through each cell).

In [ ]:
# Step 1: Verify GPU & Clone Repository
import os, pathlib, subprocess, shutil
import torch

print("=== HARDWARE CHECK ===")
if torch.cuda.is_available():
    print(f"✓ GPU Active: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ WARNING: GPU not detected! Make sure you enabled GPU accelerator in Settings!")

WORK = pathlib.Path("/kaggle/working" if os.path.exists("/kaggle") else "/content")
REPO_DIR = WORK / "the-real-houseboys-of-san-francisco"
REPO_URL = "https://github.com/alston808/the-real-houseboys-of-san-francisco.git"

if not REPO_DIR.exists():
    print("Cloning repository...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

IMAGES_DIR = REPO_DIR / "images"
AUDIO_DIR = REPO_DIR / "audio"
OUT_DIR = WORK / "outputs" / "confessionals"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Found {len(list(IMAGES_DIR.glob("*.png")))} character portraits in images/")
print(f"✓ Found {len(list(AUDIO_DIR.glob("*.wav")))} speech audio clips in audio/")
print(f"✓ Output destination: {OUT_DIR}")


In [ ]:
# Step 2: Install SadTalker (Talking Head Lip-Sync Model)
import subprocess, urllib.request

SADTALKER_DIR = WORK / "SadTalker"
if not SADTALKER_DIR.exists():
    print("Cloning SadTalker repository...")
    subprocess.run(["git", "clone", "https://github.com/OpenTalker/SadTalker.git", str(SADTALKER_DIR)], check=True)

print("Installing dependencies...")
!pip install -q torch torchvision torchaudio
!pip install -q gradio imageio[ffmpeg] yacs pydub scipy scikit-image numba resampy librosa
!pip install -q facexlib kornia torchvision fsspec packaging gfpgan

# Download model checkpoints
CHECKPOINTS_DIR = SADTALKER_DIR / "checkpoints"
CHECKPOINTS_DIR.mkdir(exist_ok=True)

def download_file(url, dest):
    if not dest.exists():
        print(f"Downloading {dest.name}...")
        urllib.request.urlretrieve(url, str(dest))

download_file(
    "https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2/SadTalker_V0.0.2_256.safetensors",
    CHECKPOINTS_DIR / "SadTalker_V0.0.2_256.safetensors"
)
download_file(
    "https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2/mapping_00109-model.pth.tar",
    CHECKPOINTS_DIR / "mapping_00109-model.pth.tar"
)
download_file(
    "https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2/mapping_00229-model.pth.tar",
    CHECKPOINTS_DIR / "mapping_00229-model.pth.tar"
)

GFPGAN_DIR = SADTALKER_DIR / "gfpgan" / "weights"
GFPGAN_DIR.mkdir(parents=True, exist_ok=True)
download_file(
    "https://github.com/xinntao/facexlib/releases/download/v0.1.0/alignment_WFLW_448.pth",
    GFPGAN_DIR / "alignment_WFLW_448.pth"
)
download_file(
    "https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth",
    GFPGAN_DIR / "GFPGANv1.4.pth"
)

print("✓ SadTalker ready!")


In [ ]:
# Step 3: Batch Generate All 18 Confessional Clips
import sys, subprocess, os, shutil, glob

CHARACTERS = ["prest", "felix", "marco", "dex", "river", "ash"]
completed_clips = []

print("=== BATCH RENDERING 18 CONFESSIONALS ===
")

for char in CHARACTERS:
    portrait_path = IMAGES_DIR / f"{char}.png"
    if not portrait_path.exists():
        print(f"⚠️ Missing portrait for {char} at {portrait_path}")
        continue

    print(f"
🎥 Processing character: {char.upper()}")
    for beat in [1, 2, 3]:
        audio_path = AUDIO_DIR / f"{char}_confessional_{beat}.wav"
        if not audio_path.exists():
            candidates = list(AUDIO_DIR.glob(f"{char}*{beat}*.wav"))
            if candidates:
                audio_path = candidates[0]
            else:
                print(f"  ⚠️ Audio not found for {char} beat {beat}")
                continue

        out_target = OUT_DIR / f"{char}_confessional_{beat}.mp4"
        if out_target.exists():
            print(f"  ✓ Already rendered: {out_target.name}")
            completed_clips.append(str(out_target))
            continue

        print(f"  -> Animating beat {beat}: {char}.png + {audio_path.name}...")
        
        cmd = [
            "python", str(SADTALKER_DIR / "inference.py"),
            "--driven_audio", str(audio_path),
            "--source_image", str(portrait_path),
            "--result_dir", str(OUT_DIR),
            "--still",
            "--preprocess", "crop",
            "--size", "256",
            "--enhancer", "gfpgan"
        ]
        
        res = subprocess.run(cmd, cwd=str(SADTALKER_DIR), capture_output=True, text=True)
        
        recent = sorted(list(OUT_DIR.glob("*.mp4")),
                        key=lambda p: os.path.getmtime(p) if p != out_target else 0)
        if recent and recent[-1] != out_target:
            shutil.move(str(recent[-1]), str(out_target))
            print(f"  ✓ Successfully created: {out_target.name}")
            completed_clips.append(str(out_target))
        elif out_target.exists():
            print(f"  ✓ Successfully created: {out_target.name}")
            completed_clips.append(str(out_target))
        else:
            print(f"  ✗ Error on {char} beat {beat}: {res.stderr[-250:]}")

print(f"
🎉 All done! Rendered {len(completed_clips)} / 18 confessional clips.")


In [ ]:
# Step 4: Zip & Download All Finished Confessionals
import shutil, os

zip_filename = WORK / "confessionals_all.zip"
print("Creating zip archive of all confessional clips...")
shutil.make_archive(str(WORK / "confessionals_all"), "zip", str(OUT_DIR))
size_mb = os.path.getsize(zip_filename) / (1024 * 1024)
print(f"✓ Archive ready: {zip_filename} ({size_mb:.2f} MB)")

# Auto download if on Google Colab
try:
    from google.colab import files
    files.download(str(zip_filename))
    print("Triggered browser download!")
except ImportError:
    print(f"On Kaggle: The zip file is in your Output tab on the right -> {zip_filename}")
